# MDR-TS v24
**Temporal-Station Baseline Model for Soil Moisture Prediction**

**Author:** Jakob Balkovec  
**Affiliation:** Seattle University, Computer Science  
**Project:** MDR
**Notebook Type:** Training & Evaluation  
**Last Updated:** Tue Jan 6th 2026

---

## Model Summary
- **Model Name:** MDR-TS  
- **Version:** v24
- **Task:** Regression (Soil Moisture at 5 cm depth)  
- **Target Variable:** `soil_moisture_5cm`  
- **Temporal Resolution:** Daily  

---

## Reproducibility
- **Random Seed:** 42
- **Split Metadata:** `data/splits/derived_9.0/split_meta.json`
- **Environment:** Google Colab / VS Code Remote Kernel

---

Adapted for a Macbook M2 Pro environment.

**What's new?**

- Adding weights to improve spatial generalization

## 0. Imports

In [25]:
import os
import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

import torch

project_root = os.path.abspath("../../")
if project_root not in sys.path:
    sys.path.append(project_root)

from Utils.dashboard import metrics_dashboard

import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("imports loaded")
print(f"using: {device}")

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))

def assign_regime(sm):
    if sm < 0.20:
        return 'D'
    elif sm < 0.313:
        return 'T'
    else:
        return 'W'

imports loaded
using: cpu


In [26]:
SEED = 42
DEEP_SEARCH = 40

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print(f"Random seed set to {SEED}")

def print_env_info():
    print("Environment information:")
    print(f"  Python version: {os.sys.version.split()[0]}")
    print(f"  NumPy version:  {np.__version__}")
    print(f"  Pandas version: {pd.__version__}")

    try:
        import xgboost
        print(f"  XGBoost version: {xgboost.__version__}")
    except ImportError:
        print("  XGBoost not installed")

    IN_COLAB = "COLAB_GPU" in os.environ
    print(f"  Running in Colab: {IN_COLAB}")

    if IN_COLAB:
        gpu = os.environ.get("COLAB_GPU", None)
        print(f"  GPU available: {gpu}")
    else:
        print("  GPU available: False")

print_env_info()

plt.style.use("default")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

print("environment setup complete")

Random seed set to 42
Environment information:
  Python version: 3.10.18
  NumPy version:  1.26.4
  Pandas version: 2.0.3
  XGBoost version: 3.2.0
  Running in Colab: False
  GPU available: False
environment setup complete


In [27]:
VERSION = "v24"
SUBVERSION = "v24"
RUN_NAME = "mdr_ts_v24"

PROJECT_ROOT = "/Users/jbalkovec/Desktop/MDR"
DATA_ROOT = f"{PROJECT_ROOT}/Temporal/Pipeline/data"
SPLIT_ROOT = f"{DATA_ROOT}/splits"
OUTPUT_ROOT = f"{PROJECT_ROOT}/Models/Temporal/{VERSION}/{SUBVERSION}"

os.makedirs(OUTPUT_ROOT, exist_ok=True)

print("Project paths:")
print(f"  PROJECT_ROOT: {PROJECT_ROOT}")
print(f"  DATA_ROOT:    {DATA_ROOT}")
print(f"  SPLIT_ROOT:   {SPLIT_ROOT}")
print(f"  OUTPUT_ROOT:  {OUTPUT_ROOT}")

print("\nKey file checks:")
print("  data exists:",
      os.path.exists(DATA_ROOT))
print("  splits exists:",
      os.path.exists(SPLIT_ROOT))
print("  output exists:",
      os.path.exists(OUTPUT_ROOT))

Project paths:
  PROJECT_ROOT: /Users/jbalkovec/Desktop/MDR
  DATA_ROOT:    /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data
  SPLIT_ROOT:   /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits
  OUTPUT_ROOT:  /Users/jbalkovec/Desktop/MDR/Models/Temporal/v24/v24

Key file checks:
  data exists: True
  splits exists: True
  output exists: True


In [28]:
TRAIN_PATH = str(Path(SPLIT_ROOT) / "derived_9.0/train.csv")
VAL_PATH   = str(Path(SPLIT_ROOT) / "derived_9.0/val.csv")
TEST_PATH  = str(Path(SPLIT_ROOT) / "derived_9.0/test.csv")

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing split file: {p}")

print("Split files:")
print(" ", TRAIN_PATH)
print(" ", VAL_PATH)
print(" ", TEST_PATH)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

Split files:
  /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits/derived_9.0/train.csv
  /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits/derived_9.0/val.csv
  /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits/derived_9.0/test.csv


In [29]:
print("Common columns across splits:",
      len(set(train_df.columns) & set(val_df.columns) & set(test_df.columns)))

print("\ncolumns:")
print(list(train_df.columns)[:30])

Common columns across splits: 499

columns:
['station_id', 'date', 'longitude', 'latitude', 'precip_mm', 's1_vv', 's1_vh', 's2_b4', 's2_b8', 's2_b11', 's2_b12', 'LST_modis', 'elev', 'slope', 'aspect', 'DOY', 'SMAP_sm_am_interp', 'SMAP_sm_pm_interp', 'soil_moisture_5cm', 'J_aspect_deg', 'J_bio_bio01', 'J_bio_bio02', 'J_bio_bio03', 'J_bio_bio04', 'J_bio_bio05', 'J_bio_bio06', 'J_bio_bio07', 'J_bio_bio08', 'J_bio_bio09', 'J_bio_bio10']


In [30]:
TARGET_COL = "soil_moisture_5cm"

KEEP_META_COLS = ["station_id", "date", "longitude", "latitude"]

FEATURE_COLS = [
    'SMAP_sm_pm_interp_ema02',
    'V_rollmin_LST_modis_kobs30',
    'D_sin_DOY', 'G_rain_sum_3d',
    'V_rollmin_G_API_kobs30',
    'G_rain_sum_7d',
    'C_lag_LST_modis_kobs30',
    'C_lag_G_API_kobs1',
    'V_ema_G_API_kobs14',
    'V_rollmean_G_API_kobs14',
    'G_API', 'G_DSLR',
    'SMAP_ampm_diff_interp',
    'V_rollmax_G_API_kobs30',
    'V_ema_G_API_kobs30',
    'V_rollmean_s2_b11_kobs7',
    'V_ema_LST_modis_kobs7',
    'V_rollmean_G_API_kobs7',
    'C_lag_s2_b11_kobs30',
    'A_d_E_SAR_diff_kobs14',
    'C_lag_LST_modis_kobs6',
    'A_d_LST_modis_kobs14',
    'A_d_SMAP_sm_interp_kobs14',
    'V_rollstd_SMAP_sm_interp_kobs30',
    'SMAP_sm_interp_grad7',
    'year_frac', 'sin_year', 'cos_year',
    'API_x_year', 'SMAP_x_year',
    'slope', 'elev', 'K_slope_sin',
    'K_slope_cos', 'K_aspect_cos',
    'J_clay_wfrac_b0', 'J_sand_wfrac_b0'
    ]

### Clusters

In [31]:
# DARRINGTON CLUSTER
# Distribution: Wet-dominated (62% Wet, 17% Transition, 20% Dry)
# Mean Soil Moisture: 0.225
# Characteristics: High moisture, low variability, mountainous/wet regions
# Use for: IN-DOMAIN validation (Darrington is training station)
#          Best for testing model on familiar wet regime
# Risk: Model may overfit to wet conditions; poor at predicting dry/transition
DARRINGTON_CLUSTER = ['Darrington',
                      'BeaverPass_WA_990',
                      'MartenRidge_WA_999',
                      'MFNooksack_WA_1011',
                      'USCRN_Darrington_21_NNE']

# QUINAULT CLUSTER
# Distribution: Transition-heavy (37% Transition, 57% Wet, 6% Dry)
# Mean Soil Moisture: 0.215
# Characteristics: Moderate moisture with high variability, balanced regimes
# Use for: MIXED spatial evaluation (Quinault is training, but Touchet is anomalous)
# Risk: Touchet is a misclassification (55.9% Transition, only 23% Wet)
#       This cluster is heterogeneous and less reliable for predictions
QUINAULT_CLUSTER = ['Quinault',
                    'Touchet_WA_824',
                    'USCRN_Corvallis_10_SSW',
                    'USCRN_Quinault_4_NE']

# SOURDOUGHGULCH CLUSTER
# Distribution: Wet-dominated (64% Wet, 23% Transition, 12% Dry)
# Mean Soil Moisture: 0.212
# Characteristics: Similar to Darrington, high moisture
# Use for: IN-DOMAIN validation (SourdoughGulch is training station)
#          Confirms wet-biased model behavior
# Risk: Only 3 OOD stations; limited power to detect spatial variance
SOURDOUGHGULCH_CLUSTER = ['SourdoughGulch_WA_985',
                          'USCRN_Dillon_18_WSW',
                          'SCAN_ConradAgRc',
                          'USCRN_Coos_Bay_8_SW']

# SPOKANE CLUSTER
# Distribution: Dry-biased (37% Dry, 20% Transition, 43% Wet) - OUTLIER REGIME
# Mean Soil Moisture: 0.129 (LOWEST of all clusters)
# Characteristics: Semi-arid to arid; represents dry interior Washington
# Use for: OUT-OF-DOMAIN stress test (Spokane is training, but represents minority regime)
#          This is where the model will likely FAIL most
# Risk: LARGE domain shift - 17 OOD stations heavily skew dry
#       Model trained on 53% wet, but these stations are 44% dry on average
#       Expect significant performance degradation here
# Recommendation: AVOID for in-domain validation; USE for identifying spatial weakness
SPOKANE_CLUSTER = ['Spokane',
                   'CayusePass_WA',
                   'Paradise_WA',
                   'BurntMountain_WA',
                   'HartsPass_WA_515',
                   'RainyPass_WA_711',
                   'USCRN_Arco_17_SW',
                   'USCRN_John_Day_35_WNW',
                    'USCRN_Murphy_10_W',
                    'USCRN_Riley_10_WSW',
                    'USCRN_Spokane_17_SSW',
                    'USCRN_St_Mary_1_SSW',
                    'SCAN_CookFarmFieldD',
                    'SCAN_JordanValleyCwma',
                    'SCAN_Lind_1',
                    'SCAN_OrchardRangeSite',
                    'SCAN_TableMountain',
                    'SCAN_Violett']

# Baseline
ORIGINAL_CLUSTER = ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane', "Touchet_WA_824"]

ALL_CLUSTERS = {
    'Darrington': DARRINGTON_CLUSTER,
    'Quinault': QUINAULT_CLUSTER,
    'SourdoughGulch_WA_985': SOURDOUGHGULCH_CLUSTER,
    'Spokane': SPOKANE_CLUSTER,
    'Original': ORIGINAL_CLUSTER,
}

In [32]:
def get_cluster_splits(train_df, val_df, test_df, cluster_name):
    if cluster_name not in ALL_CLUSTERS:
        raise ValueError(f"Cluster '{cluster_name}' not found. Choose from: {list(ALL_CLUSTERS.keys())}")

    cluster_stations = ALL_CLUSTERS[cluster_name]

    cluster_train = train_df[train_df['station_id'].isin(cluster_stations)]
    cluster_val = val_df[val_df['station_id'].isin(cluster_stations)]
    cluster_test = test_df[test_df['station_id'].isin(cluster_stations)]

    return {
        'train': cluster_train,
        'val': cluster_val,
        'test': cluster_test,
        'cluster_name': cluster_name,
        'stations': cluster_stations,
    }

### Interaction Features

In [33]:
corr = train_df[FEATURE_COLS].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

high_corr = upper.stack()
high_corr = high_corr[high_corr > 0.995]

if high_corr.empty:
    print("No highly correlated pairs found")
else:
    print("Highly correlated pairs:")
    for (col, row), val in high_corr.items():
        print(f"{col} <-> {row} : {val:.5f}")

Highly correlated pairs:
C_lag_G_API_kobs1 <-> G_API : 0.99966
C_lag_G_API_kobs1 <-> V_rollmean_G_API_kobs7 : 0.99888
V_ema_G_API_kobs14 <-> V_rollmean_G_API_kobs14 : 0.99969
V_ema_G_API_kobs14 <-> V_rollmean_G_API_kobs7 : 0.99782
V_rollmean_G_API_kobs14 <-> V_rollmean_G_API_kobs7 : 0.99729
G_API <-> V_rollmean_G_API_kobs7 : 0.99762


In [34]:
def get_metrics_dict(y_true, y_pred, prefix=""):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    err = y_true - y_pred
    ae = np.abs(err)

    r2 = float(r2_score(y_true, y_pred))
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(root_mean_squared_error(y_true, y_pred))

    bias = float(np.mean(err))
    ubrmse = float(np.std(err))

    q_err = np.quantile(err, [0.05, 0.25, 0.50, 0.75, 0.95])

    return {
        f"{prefix}n": int(len(y_true)),
        f"{prefix}r2": r2,
        f"{prefix}mae": mae,
        f"{prefix}rmse": rmse,
        f"{prefix}ubrmse": ubrmse,
        f"{prefix}bias": bias,
        f"{prefix}med_ae": float(np.median(ae)),
        f"{prefix}p90_ae": float(np.quantile(ae, 0.90)),
        f"{prefix}q05_err": float(q_err[0]),
        f"{prefix}q50_err": float(q_err[2]),
        f"{prefix}q95_err": float(q_err[4]),
    }

def neat_print(metrics):
    print(f"{'METRIC':<15} | {'VALUE':<10}")
    print("-" * 28)
    for k, v in metrics.items():
        val_str = f"{v:,}" if isinstance(v, int) else f"{v:+.5f}"
        print(f"{k:<15} | {val_str:<10}")

### Split Strategy
- **Training set:**  
  Two stations, early time period  
- **Validation set:**  
  Same stations as training, held-out **future dates** (temporal holdout)
- **Test set:**  
  One completely unseen station (station-level holdout)

### Motivation
- Validation evaluates **temporal generalization** on known stations
- Test evaluates **spatial generalization** to an unseen station
- This avoids spatial leakage while preserving sufficient training data

In [35]:
def split_summary(name, d):
    print(f"\n{name.upper()}")
    print(f"  rows:     {len(d)}")
    print(f"  stations: {sorted(d['station_id'].unique().tolist())}")
    if "date" in d.columns:
        print(f"  date range: {d['date'].min()} -- {d['date'].max()}")

print("=== ORIGINAL TEMPORAL SPLITS ===")
split_summary("train", train_df)
split_summary("val", val_df)
split_summary("test", test_df)

print("\n=== CLUSTER-BASED SPLITS ===\n")

cluster_splits = {}
for cluster_name in ALL_CLUSTERS.keys():
    cluster_splits[cluster_name] = get_cluster_splits(train_df, val_df, test_df, cluster_name)

    splits = cluster_splits[cluster_name]
    print(f"\n{cluster_name.upper()} CLUSTER SPLITS:")
    split_summary("train", splits['train'])
    split_summary("val", splits['val'])
    split_summary("test", splits['test'])

print("\n=== LEAKAGE CHECK (original splits) ===")
print("train ∩ test:", sorted(set(train_df.station_id) & set(test_df.station_id)))
print("val   ∩ test:", sorted(set(val_df.station_id) & set(test_df.station_id)))

print("\n-- splits locked --")

# get spokane test split
# spokane_test = cluster_splits['Spokane']['test']

# get all darrington data
# darrington_data = {
#     'train': cluster_splits['Darrington']['train'],
#     'val': cluster_splits['Darrington']['val'],
#     'test': cluster_splits['Darrington']['test'],
# }

=== ORIGINAL TEMPORAL SPLITS ===

TRAIN
  rows:     29362
  stations: ['BeaverPass_WA_990', 'BurntMountain_WA', 'CayusePass_WA', 'Darrington', 'HartsPass_WA_515', 'MFNooksack_WA_1011', 'MartenRidge_WA_999', 'Paradise_WA', 'Quinault', 'RainyPass_WA_711', 'SCAN_ConradAgRc', 'SCAN_CookFarmFieldD', 'SCAN_JordanValleyCwma', 'SCAN_Lind_1', 'SCAN_OrchardRangeSite', 'SCAN_TableMountain', 'SCAN_Violett', 'SourdoughGulch_WA_985', 'Spokane', 'Touchet_WA_824', 'USCRN_Arco_17_SW', 'USCRN_Corvallis_10_SSW', 'USCRN_Darrington_21_NNE', 'USCRN_Dillon_18_WSW', 'USCRN_John_Day_35_WNW', 'USCRN_Murphy_10_W', 'USCRN_Quinault_4_NE', 'USCRN_Riley_10_WSW', 'USCRN_Spokane_17_SSW', 'USCRN_St_Mary_1_SSW']
  date range: 2017-01-01 -- 2020-12-31

VAL
  rows:     13637
  stations: ['BeaverPass_WA_990', 'BurntMountain_WA', 'CayusePass_WA', 'Darrington', 'HartsPass_WA_515', 'MartenRidge_WA_999', 'Paradise_WA', 'Quinault', 'RainyPass_WA_711', 'SCAN_ConradAgRc', 'SCAN_CookFarmFieldD', 'SCAN_JordanValleyCwma', 'SCAN_Lind

In [36]:
def evaluate_cluster(cluster_name, train_df, val_df, test_df,
                     FEATURE_COLS, TARGET_COL, SEED=42):

    splits = get_cluster_splits(train_df, val_df, test_df, cluster_name)
    train_c = splits['train']
    val_c = splits['val']
    test_c = splits['test']

    print("=" * 70)
    print(f"CLUSTER EVALUATION: {cluster_name.upper()}")
    print("=" * 70)
    print(f"Stations in cluster: {len(splits['stations'])}")
    print(f"  {', '.join(splits['stations'])}")
    print()

    # combine trainval
    trainval_c = pd.concat([train_c, val_c], axis=0).reset_index(drop=True)

    # temporal weighting
    trainval_c["date"] = pd.to_datetime(trainval_c["date"], errors="coerce")
    trainval_c["year"] = trainval_c["date"].dt.year.astype(float)
    max_year = trainval_c["year"].max()
    beta = 0.2

    w_trainval = np.exp(beta * (trainval_c["year"] - max_year))
    w_trainval = w_trainval / w_trainval.mean()

    # regime weighting
    trainval_c['regime'] = trainval_c[TARGET_COL].apply(assign_regime)
    regime_counts = trainval_c['regime'].value_counts()
    regime_weights = 1.0 / regime_counts
    regime_weights = regime_weights / regime_weights.sum()

    w_regime = trainval_c['regime'].map(regime_weights)
    w_combined = (w_trainval * w_regime)
    w_combined = w_combined / w_combined.mean()

    # prepare data
    X_trainval = trainval_c[FEATURE_COLS].copy()
    y_trainval = trainval_c[TARGET_COL].copy()
    X_test = test_c[FEATURE_COLS].copy()
    y_test = test_c[TARGET_COL].copy()

    print(f"Train+Val samples: {len(X_trainval)}")
    print(f"Test samples:      {len(X_test)}")
    print()

    # RF params
    RF_PARAMS = dict(
        n_estimators=1000,
        max_depth=28,
        min_samples_split=4,
        min_samples_leaf=1,
        max_features="sqrt",
        random_state=SEED,
        n_jobs=-1,
    )

    # train with combined weights
    rf_model = RandomForestRegressor(**RF_PARAMS)
    rf_model.fit(X_trainval, y_trainval, sample_weight=w_combined)

    # predict
    y_pred = rf_model.predict(X_test).ravel()
    y_test_arr = np.asarray(y_test).ravel()

    # metrics
    metrics = get_metrics_dict(y_test_arr, y_pred, prefix=f"{cluster_name}_")

    print("=" * 70)
    print(f"TEST PERFORMANCE: {cluster_name}")
    print("=" * 70)
    neat_print(metrics)

    print()
    metrics_dashboard(y_test_arr, y_pred, name=f"Test Set - {cluster_name}", return_dict=False)

    return {
        'cluster_name': cluster_name,
        'metrics': metrics,
        'y_test': y_test_arr,
        'y_pred': y_pred,
        'model': rf_model,
        'splits': splits,
        'n_samples': len(X_test),
        'n_stations': len(splits['stations']),
    }

In [37]:
results = {}
for cluster_name in ALL_CLUSTERS.keys():
    result = evaluate_cluster(cluster_name, train_df, val_df, test_df,
                              FEATURE_COLS, TARGET_COL)
    results[cluster_name] = result
    print("\n\n")

CLUSTER EVALUATION: DARRINGTON
Stations in cluster: 5
  Darrington, BeaverPass_WA_990, MartenRidge_WA_999, MFNooksack_WA_1011, USCRN_Darrington_21_NNE

Train+Val samples: 5630
Test samples:      2628

TEST PERFORMANCE: Darrington
METRIC          | VALUE     
----------------------------
Darrington_n    | 2,628     
Darrington_r2   | +0.12583  
Darrington_mae  | +0.07029  
Darrington_rmse | +0.09511  
Darrington_ubrmse | +0.08880  
Darrington_bias | -0.03406  
Darrington_med_ae | +0.04906  
Darrington_p90_ae | +0.17636  
Darrington_q05_err | -0.19453  
Darrington_q50_err | -0.01782  
Darrington_q95_err | +0.10085  






CLUSTER EVALUATION: QUINAULT
Stations in cluster: 4
  Quinault, Touchet_WA_824, USCRN_Corvallis_10_SSW, USCRN_Quinault_4_NE

Train+Val samples: 7581
Test samples:      2558

TEST PERFORMANCE: Quinault
METRIC          | VALUE     
----------------------------
Quinault_n      | 2,558     
Quinault_r2     | +0.25486  
Quinault_mae    | +0.04889  
Quinault_rmse   | +0.06916  
Quinault_ubrmse | +0.06601  
Quinault_bias   | +0.02063  
Quinault_med_ae | +0.03239  
Quinault_p90_ae | +0.12681  
Quinault_q05_err | -0.10168  
Quinault_q50_err | +0.01695  
Quinault_q95_err | +0.14056  






CLUSTER EVALUATION: SOURDOUGHGULCH_WA_985
Stations in cluster: 4
  SourdoughGulch_WA_985, USCRN_Dillon_18_WSW, SCAN_ConradAgRc, USCRN_Coos_Bay_8_SW

Train+Val samples: 4915
Test samples:      2218

TEST PERFORMANCE: SourdoughGulch_WA_985
METRIC          | VALUE     
----------------------------
SourdoughGulch_WA_985_n | 2,218     
SourdoughGulch_WA_985_r2 | -0.79564  
SourdoughGulch_WA_985_mae | +0.08001  
SourdoughGulch_WA_985_rmse | +0.11691  
SourdoughGulch_WA_985_ubrmse | +0.10749  
SourdoughGulch_WA_985_bias | +0.04597  
SourdoughGulch_WA_985_med_ae | +0.04881  
SourdoughGulch_WA_985_p90_ae | +0.19109  
SourdoughGulch_WA_985_q05_err | -0.09554  
SourdoughGulch_WA_985_q50_err | +0.02295  
SourdoughGulch_WA_985_q95_err | +0.27708  






CLUSTER EVALUATION: SPOKANE
Stations in cluster: 18
  Spokane, CayusePass_WA, Paradise_WA, BurntMountain_WA, HartsPass_WA_515, RainyPass_WA_711, USCRN_Arco_17_SW, USCRN_John_Day_35_WNW, USCRN_Murphy_10_W, USCRN_Riley_10_WSW, USCRN_Spokane_17_SSW, USCRN_St_Mary_1_SSW, SCAN_CookFarmFieldD, SCAN_JordanValleyCwma, SCAN_Lind_1, SCAN_OrchardRangeSite, SCAN_TableMountain, SCAN_Violett

Train+Val samples: 24873
Test samples:      12762

TEST PERFORMANCE: Spokane
METRIC          | VALUE     
----------------------------
Spokane_n       | 12,762    
Spokane_r2      | +0.46206  
Spokane_mae     | +0.05489  
Spokane_rmse    | +0.07755  
Spokane_ubrmse  | +0.07733  
Spokane_bias    | +0.00579  
Spokane_med_ae  | +0.03664  
Spokane_p90_ae  | +0.13107  
Spokane_q05_err | -0.11606  
Spokane_q50_err | -0.00008  
Spokane_q95_err | +0.14702  






CLUSTER EVALUATION: ORIGINAL
Stations in cluster: 5
  Darrington, Quinault, SourdoughGulch_WA_985, Spokane, Touchet_WA_824

Train+Val samples: 9588
Test samples:      4016

TEST PERFORMANCE: Original
METRIC          | VALUE     
----------------------------
Original_n      | 4,016     
Original_r2     | +0.83280  
Original_mae    | +0.02822  
Original_rmse   | +0.03850  
Original_ubrmse | +0.03850  
Original_bias   | -0.00010  
Original_med_ae | +0.02154  
Original_p90_ae | +0.05923  
Original_q05_err | -0.06116  
Original_q50_err | -0.00012  
Original_q95_err | +0.05821  



In [39]:
print("=" * 70)
print("CLUSTER PERFORMANCE COMPARISON")
print("=" * 70)
print()

comparison_data = []
for cluster_name, result in results.items():
    r2 = result['metrics'][f'{cluster_name}_r2']
    rmse = result['metrics'][f'{cluster_name}_rmse']
    mae = result['metrics'][f'{cluster_name}_mae']
    n_samples = result['n_samples']
    n_stations = result['n_stations']

    comparison_data.append({
        'Cluster': cluster_name,
        'R2': r2,
        'RMSE': rmse,
        'MAE': mae,
        'Samples': n_samples,
        'Stations': n_stations,
    })

comparison_df = pd.DataFrame(comparison_data).sort_values('R2', ascending=False)
print(comparison_df.to_string(index=False))

print("\n" + "=" * 70)
print("INTERPRETATION")
print("=" * 70)
best_cluster = comparison_df.iloc[0]['Cluster']
worst_cluster = comparison_df.iloc[-1]['Cluster']
print(f"Best performing: {best_cluster}")
print(f"Worst performing: {worst_cluster}")

CLUSTER PERFORMANCE COMPARISON

              Cluster        R2     RMSE      MAE  Samples  Stations
             Original  0.832805 0.038503 0.028216     4016         5
              Spokane  0.462057 0.077547 0.054887    12762        18
             Quinault  0.254856 0.069156 0.048894     2558         4
           Darrington  0.125826 0.095108 0.070290     2628         5
SourdoughGulch_WA_985 -0.795645 0.116910 0.080015     2218         4

INTERPRETATION
Best performing: Original
Worst performing: SourdoughGulch_WA_985


---

_Jakob Balkovec_